**Лабораторный практикум по курсу «Распознавание диктора», Университет ИТМО, 2021**		

**Лабораторная работа №3. Построение дикторских моделей и их сравнение**

**Цель работы:** изучение процедуры построения дикторских моделей с использованием глубоких нейросетевых архитектур.

**Краткое описание:** в рамках настоящей лабораторной работы предлагается изучить и реализовать схему построения дикторских моделей с использованием глубокой нейросетевой архитектуры, построенной на основе ResNet-блоков. Процедуры обучения и тестирования предлагается рассмотреть по отношению к задаче идентификации на закрытом множестве, то есть для ситуации, когда дикторские классы являются строго заданными. Тестирование полученной системы предполагает использование доли правильных ответов (accuracy) в качестве целевой метрики оценки качества.

**Данные:** в качестве данных для выполнения лабораторной работы предлагается использовать базу [VoxCeleb1](http://www.robots.ox.ac.uk/~vgg/data/voxceleb/vox1.html).

**Содержание лабораторной работы**

1. Подготовка данных для обучения и тестирования блока построения дикторских моделей.							

2. Обучение параметров блока построения дикторских моделей без учёта процедуры аугментации данных.

3. Обучение параметров блока построения дикторских моделей с учётом процедуры аугментации данных.

4. Тестирование блока построения дикторских моделей.

In [ ]:
# Автоматически перезагружаем изменённые модули перед выполнением кода
%load_ext autoreload
%autoreload 2

# Импортируем необходимые модули
import os
import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve()))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from common import (
    concatenate,
    download_dataset,
    download_protocol,
    extract_dataset,
    part_extract,
    split_musan,
)
from constants import (
    ANGULAR_MARGIN,
    ANGULAR_SCALE,
    AUGMENTATION_DATASET_LIST_PATH,
    CHECKPOINT_FILENAME_TEMPLATE,
    CHECKPOINT_RESUME_ENABLED,
    CHECKPOINT_RESUME_EPOCH,
    CONCATENATION_LIST_PATH,
    DATA_DIR,
    DATASET_LIST_PATH,
    DEFAULT_AUGMENTED_CHECKPOINT_DIR,
    DEFAULT_CHECKPOINT_DIR,
    DEFAULT_VOXCELEB_DEV_DIR,
    EMBEDDING_SIZE,
    ENCODER_STATISTICAL_POOLING,
    EPOCH_RESULT_MESSAGE,
    ERROR_VOXCELEB_CREDENTIALS,
    EVAL_BATCH_SIZE,
    EVAL_MAX_FRAMES,
    EVAL_WORKERS,
    FULL_TEST_RESULT_MESSAGE,
    LEARNING_RATE,
    MAX_EPOCHS,
    MEL_FILTERS,
    MODEL_WITH_AUGMENTATION,
    MODEL_WITHOUT_AUGMENTATION,
    MUSAN_ARCHIVE_PATH,
    MUSAN_PATH,
    ONE_CYCLE_DIV_FACTOR,
    ONE_CYCLE_FINAL_DIV_FACTOR,
    ONE_CYCLE_START_FRACTION,
    PIN_MEMORY,
    PROTOCOL_LIST_PATH,
    RESNET_FILTERS,
    RESNET_LAYERS,
    RIR_ARCHIVE_PATH,
    RIR_EXTRACTION_TARGETS,
    RIR_PATH,
    TEXT_ENCODING,
    TRAIN_BATCH_SIZE,
    TRAIN_MAX_FRAMES,
    TRAIN_SHUFFLE,
    TRAIN_WORKERS,
    VALIDATION_INTERVAL,
    VALIDATION_RESULT_MESSAGE,
    VOXCELEB_DEV_ARCHIVE_PATH,
    VOXCELEB_DEV_EXTRACT_PATH,
    VOXCELEB_PASSWORD_ENV,
    VOXCELEB_TEST_DOWNLOAD_PATH,
    VOXCELEB_USER_ENV,
    WEIGHT_DECAY,
)
from exercises_blank import (
    MainModel,
    ResNet,
    test_dataset_loader,
    test_network,
    train_dataset_loader,
    train_network,
)
from load_save_pth import loadParameters, saveParameters
from LossFunction import AAMSoftmaxLoss
from main import build_protocol_lists
from Optimizer import SGDOptimizer
from ResNetBlocks import BasicBlock
from Scheduler import OneCycleLRScheduler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

**1. Подготовка данных для обучения и тестирования дикторской модели**

В ходе выполнения лабораторной работы необходимы данные для выполнения процедуры обучения и процедуры тестирования нейросетевого блока генерации дикторских моделей. Возьмём в качестве этих данных звукозаписи, сохраненные в формат *wav*, из корпуса [VoxCeleb1 dev set](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/vox1.html). Данный корпус содержит 148,642 звукозаписи (частота дискретизации равна 16кГц) для 1,211 дикторов женского и мужского пола, разговаривающих преимущественно на английском языке.

В рамках настоящего пункта требуется выполнить загрузку и распаковку звуковых wav-файлов из корпуса VoxCeleb1 dev set.

![Рисунок 1](../images/voxceleb_dataset.png "VoxCeleb. Крупномасштабная аудиовизуальная база данных человеческой речи.")

In [ ]:
# Загружаем VoxCeleb1. Данные доступа не храним в ноутбуке.
voxceleb_user = os.environ.get(VOXCELEB_USER_ENV)
voxceleb_password = os.environ.get(VOXCELEB_PASSWORD_ENV)
if not voxceleb_user or not voxceleb_password:
    raise RuntimeError(ERROR_VOXCELEB_CREDENTIALS)

with DATASET_LIST_PATH.open('r', encoding=TEXT_ENCODING) as dataset_list_file:
    dataset_lines = dataset_list_file.readlines()

download_dataset(
    dataset_lines,
    user=voxceleb_user,
    password=voxceleb_password,
    save_path=str(DATA_DIR),
)

In [ ]:
# Объединяем части архива VoxCeleb1 dev set
with CONCATENATION_LIST_PATH.open('r', encoding=TEXT_ENCODING) as archive_list_file:
    archive_lines = archive_list_file.readlines()

concatenate(archive_lines, save_path=str(DATA_DIR))

In [ ]:
# Распаковываем VoxCeleb1 dev set
extract_dataset(
    save_path=str(VOXCELEB_DEV_EXTRACT_PATH),
    fname=str(VOXCELEB_DEV_ARCHIVE_PATH),
)

In [ ]:
# Загружаем протокол идентификации VoxCeleb1
with PROTOCOL_LIST_PATH.open('r', encoding=TEXT_ENCODING) as protocol_list_file:
    protocol_lines = protocol_list_file.readlines()

download_protocol(protocol_lines, save_path=str(VOXCELEB_TEST_DOWNLOAD_PATH))

**2. Обучение параметров блока построения дикторских моделей без учёта процедуры аугментации данных**

Построение современных дикторских моделей, как правило, выполняется с использованием нейросетевых архитектур, многие из которых позаимствованы из области обработки цифровых изображений. Одними из наиболее распространенных нейросетевых архитектур, используемыми для построения дикторских моделей, являются [ResNet-подобные архитектуры](https://arxiv.org/pdf/1512.03385.pdf). В рамках настоящего пункта предлагается выполнить адаптацию нейросетевой архитектуры ResNet34 для решения задачи генерации дикторских моделей (дикторских эмбеддингов). *Дикторский эмбеддинг* – это высокоуровневый вектор-признаков, состоящий, например, из 128, 256 и т.п. значений, содержащий особенности голоса конкретного человека. При решении задачи распознавания диктора можно выделить эталонные и тестовые дикторские эмбеддинги. *Эталонные эмбеддинги* формируются на этапе регистрации дикторской модели определённого человека и находятся в некотором хранилище данных. *Тестовые эмбеддинги* формируются на этапе непосредственного использования системы голосовой биометрии на практике, когда некоторый пользователь пытается получить доступ к соответствующим ресурсам. Система голосовой биометрии сравнивает по определённой метрике эталонные и тестовые эмбеддинги, формируя оценку сравнения, которая, после её обработки блоком принятия решения, позволяет сделать вывод о том, эмбеддинги одинаковых или разных дикторов сравниваются между собой.

Адаптация различных нейросетевых архитектур из обработки изображений к решению задачи построения дикторских моделей является непростой задачей. Возьмём за основу готовое решение, предложенной в рамках [следующей публикации](https://arxiv.org/pdf/2002.06033.pdf) и адаптируем его применительно к выполнению настоящей лабораторной работы.

Необходимо отметить, что построение дикторских моделей, как правило, требует наличия *акустических признаков*, вычисленных для звукозаписей тренировочной, валидационной и тестовой баз данных. В качестве примера подобных признаков в рамках настоящей лабораторной работы воспользуемся *логарифмами энергий на выходе мел-банка фильтров*. Важно отметить, что акустические признаки подвергаются некоторым процедурам предобработки перед их непосредственной передачей в блок построения дикторских моделей. В качестве этих процедур можно выделить: нормализация и масштабирование признаков, сохранение только речевых фреймов на основе разметки детектора речевой активности и т.п.

После того, как акустические признаки подготовлены, они могут быть переданы на блок построения дикторских моделей. Как правило, структура современных дикторских моделей соответствует структуре [x-векторных архитектур](https://www.danielpovey.com/files/2018_icassp_xvectors.pdf). Эти архитектуры состоят из четырёх ключевых элементов: 

1. **Фреймовый уровень.** Предназначен для формирования локальных представлений голоса конкретного человека. На этом уровне как раз и применяются нейросетевые архитектуры на базе свёрточных нейронных сетей, например, ResNet, позволяющих с использованием каскадной схемы из множества фильтров с локальной маской захватить некоторый локальный контекст шаблона голоса человека. Выходом фреймового уровня является набор высокоуровневых представлений (карт-признаков), содержащих локальные особенности голоса человека.

2. **Уровень статистического пулинга** позволяет сформировать промежуточный вектор-признаков, фиксированной длины, которая является одинаковой для звукозаписи любой длительности. В ходе работы блока статистического пулинга происходит удаление временной размерности, присутствующей в картах-признаков. Это достигается путём выполнения процедуры усреднения карт-признаков вдоль оси времени. Выходом уровня статистического пулинга являются вектор среднего и вектор среднеквадратического отклонения, вычисленные на основе карт-признаков. Эти вектора конкатенируются и передаются для дальнейшей обработки на сегментом уровне.

3. **Сегментный уровень.** Предназначен для трансформации промежуточного вектора, как правило, высокой размерности, в компактный вектор-признаков, представляющий собой дикторский эмбеддинг. Необходимо отметить, что на сегментном уровне расположены один или несколько полносвязных нейросетевых слоёв, а обработка данных выполняется по отношению ко всей звукозаписи, а не только к некоторому её локальному контексту, как на фреймовом уровне.

4. **Уровень выходного слоя.** Представляет полносвязный слой с softmax-функциями активации. Количество активаций равно числу дикторов в тренирочной выборке. На вход выходноя слоя подаётся дикторский эмбеддинг, а на выходе – формируется набор апостериорных вероятностей, определяющих принадлежность эмбеддинга к одному из дикторских классов в тренировочной выборке. Необходимо отметить, что, как правило, в современных нейросетевых системах построения дикторских моделей выходной используется только на этапе обучения параметров и на этапе тестирования не используется (на этапе тестирования используются только три первых уровня архитектуры).

Обучение модели генерации дикторских эмбеддингов выполняется путём решения задачи *классификации* или, выражаясь терминами из области биометрии, *идентификации на закрытом множестве* (количество дикторских меток является строго фиксированным). В качестве используемой стоимостной функции выступает *категориальная кросс-энтропия*. Обучение выполняется с помощью мини-батчей, содержащих короткие фрагменты карт акустических признаков (длительностью несколько секунд) различных дикторов из тренировочной базы данных. Обучение на коротких фрагментов позволяет избежать сильного переобучения нейросетевой модели. При выполнении процедуры обучения требуется подобрать набор гиперпараметров, выбрать обучения и метод численной оптимизации.

Для успешного выполнения настоящего пункта необходимо сделать следующее:

1. Сгенерировать списки тренировочных, валидационных и тестовых данных на основе идентификационного протокола базы VoxCeleb1, содержащегося в файле **../data/voxceleb1_test/iden_split.txt**. При генерации списков требуется исключить из них звукозаписи дикторов, которые входят в базу [VoxCeleb1 test set](https://thor.robots.ox.ac.uk/~vgg/data/voxceleb/vox1a/vox1_test_wav.zip). Это позволит выполнить тестирования обученных блоков генерации дикторских моделей на протоколе [VoxCeleb1-O cleaned](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/meta/veri_test2.txt), который составлен по отношению к данным из VoxCeleb1 test set, в лабораторной работе №4.

2. Инициализировать обучаемую дикторскую модель, выбрав любой возможный вариант её архитектуры, предлагаемый в рамках лабораторной работы. При реализации блока статистического пулинга предлагается выбрать либо его классический вариант, предложенный в [следующей работе](https://www.danielpovey.com/files/2018_icassp_xvectors.pdf), либо его более продвинутую версию основанную на использовании [механизмов внимания](https://arxiv.org/pdf/1803.10963.pdf). Использование последней версии статистического пулинга позволяет реализовать детектор речевой активности прямо внутри блока построения дикторских моделей.

3. Инициализировать загрузчики тренировочной и валидационной выборки.

4. Инициализировать оптимизатор и планировщик для выполнения процедуры обучения.

5. Описать процедуру валидации/тестирования блока построения дикторских моделей.

6. Описать процедуру обучения и запустить её, контролируя значения стоимостной функции и доли правильных ответов на тренировочном множестве, а также долю правильных ответов на валидационном множестве.

In [ ]:
# Выбираем гиперпараметры из общего файла констант
n_mels = MEL_FILTERS
log_input = True
layers = RESNET_LAYERS
activation = nn.ReLU
num_filters = RESNET_FILTERS
encoder_type = ENCODER_STATISTICAL_POOLING
embedding_size = EMBEDDING_SIZE
margin = ANGULAR_MARGIN
scale = ANGULAR_SCALE

max_frames_train = TRAIN_MAX_FRAMES
train_path = DEFAULT_VOXCELEB_DEV_DIR
batch_size_train = TRAIN_BATCH_SIZE
pin_memory = PIN_MEMORY
num_workers_train = TRAIN_WORKERS
shuffle = TRAIN_SHUFFLE

max_frames_eval = EVAL_MAX_FRAMES
batch_size_eval = EVAL_BATCH_SIZE
num_workers_eval = EVAL_WORKERS

learning_rate = LEARNING_RATE
weight_decay = WEIGHT_DECAY
validation_interval = VALIDATION_INTERVAL
max_epoch = MAX_EPOCHS

In [ ]:
# Строим тренировочный, валидационный и тестовый списки по протоколу
train_list, validation_list, test_list, number_of_train_speakers = build_protocol_lists()

In [ ]:
# Инициализируем модель без жёсткой привязки к CUDA
speaker_model = ResNet(
    BasicBlock,
    layers=layers,
    activation=activation,
    num_filters=num_filters,
    nOut=embedding_size,
    encoder_type=encoder_type,
    n_mels=n_mels,
    log_input=log_input,
)
loss_function = AAMSoftmaxLoss(
    nOut=embedding_size,
    nClasses=number_of_train_speakers,
    margin=margin,
    scale=scale,
)
main_model = MainModel(speaker_model, loss_function).to(device)

In [ ]:
# Создаём загрузчик тренировочных данных без аугментации
train_dataset = train_dataset_loader(
    train_list=train_list,
    max_frames=max_frames_train,
    train_path=str(train_path),
)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size_train,
    pin_memory=pin_memory,
    num_workers=num_workers_train,
    shuffle=shuffle,
)

# Создаём загрузчик валидационных данных
validation_dataset = test_dataset_loader(
    test_list=validation_list,
    max_frames=max_frames_eval,
    test_path=str(train_path),
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size_eval,
    num_workers=num_workers_eval,
)

In [ ]:
# Инициализируем оптимизатор и планировщик скорости обучения
optimizer = SGDOptimizer(
    main_model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)
scheduler = OneCycleLRScheduler(
    optimizer,
    pct_start=ONE_CYCLE_START_FRACTION,
    cycle_momentum=False,
    max_lr=learning_rate,
    div_factor=ONE_CYCLE_DIV_FACTOR,
    final_div_factor=ONE_CYCLE_FINAL_DIV_FACTOR,
    total_steps=max_epoch * len(train_loader),
)

In [ ]:
start_epoch = 0
if CHECKPOINT_RESUME_ENABLED:
    checkpoint_path = DEFAULT_CHECKPOINT_DIR / CHECKPOINT_FILENAME_TEMPLATE.format(
        epoch=CHECKPOINT_RESUME_EPOCH,
    )
    start_epoch = loadParameters(main_model, optimizer, scheduler, checkpoint_path) + 1

# Обучаем модель без аугментации
for epoch in range(start_epoch, max_epoch):
    train_loss, train_accuracy = train_network(
        train_loader,
        main_model,
        optimizer,
        scheduler,
        epoch + 1,
        verbose=True,
    )
    print(
        EPOCH_RESULT_MESSAGE.format(
            epoch=epoch + 1,
            loss=train_loss,
            accuracy=train_accuracy,
        )
    )

    if (epoch + 1) % validation_interval == 0:
        _, validation_accuracy = test_network(validation_loader, main_model)
        print(
            VALIDATION_RESULT_MESSAGE.format(
                epoch=epoch + 1,
                accuracy=validation_accuracy,
            )
        )
        saveParameters(
            main_model,
            optimizer,
            scheduler,
            epoch,
            DEFAULT_CHECKPOINT_DIR,
        )

**3. Обучение параметров блока построения дикторских моделей с учётом процедуры аугментации данных**

Известно, что процедуры формирования и передачи речевого сигнала могут сопровождаться воздействием шумов и помех, приводящих к искажению сигнала. В качестве примеров искажающих факторов, влияющих на ухудшение качестве речевого сигнала можно привести: импульсный отклик помещения (реверберация), фоновый шум голосов группы нецелевых дикторов, звук телевизора или радиоприёмника и т.п. Разработка конвейера системы голосовой биометрии требует учёта воздействия искажающих факторов на качество её работы. Поскольку процедура построения современных дикторских моделей основана на обучении глубоких нейронных сетей, требующих большие объёмы данных для обучения их параметров, возможным вариантом увеличения тренировочной выборки может являться использование методов аугментации статистических данных. *Аугментация* – методика создания дополнительных обучающих примеров из имеющихся данных путём внесения в них искажений, которые могут потенциально возникнуть на этапе итогового тестирования системы.

Как правило, при решении задачи аугментации данных в речевой обработке используются дополнительные базы шумов и помех. В качестве примеров можно привести базы [SLR17](https://openslr.org/17/) (корпус музыкальных, речевых и шумовых звукозаписей) и [SLR28](https://openslr.org/28/) (база данных реальных и симулированных импульсных откликов комнат, а также изотропных и точечных шумов). Важно отметить, что перед применением с использованием методов аугментации подобных баз к имеющимся данным, требуется убедиться, что частоты дискретизации искажающих баз и оригинальных данных являются одинаковыми. Применительно к рассматриваемому лабораторному практикуму частоты дискретизации всех используемых звукозаписей должны быть равными 16кГц.

Как известно, можно выделить два режима аугментации данных: *онлайн* (применяется в ходе процедуры обучения) и *оффлайн* (применяется до процедуры обучения) аугментацию. В рамках настоящей лабораторной работы предлагается использовать онлайн-аугментацию из-за сравнительно небольшого набора тренировочных данных и большей гибкости экспериментов, чем в случае оффлайн-аугментации. Необходимо отметить, что применение онлайн-аугментации на практике замедляет процедуру обучения по сравнению с оффлайн-аугментацией, так как наложение искажений, извлечение акустических признаков и их возможная предобработка требуют вычислительного времени.

В рамках настоящего пункта предлагается сделать следующее:

1. Загрузить и извлечь данные из базы SLR17 (MUSAN). Частота дискретизации данных в рассматриваемой базе равна 16кГц по умолчанию. Поскольку звукозаписи рассматриваемой базы являются достаточно длинными, рекомендуется предварительно разбить эту базу на более маленькие фрагменты (например, длительностью 5 секунд с шагом 3 секунды), сохранив их на диск. 

2. Загрузить и извлечь данные из базы SLR28 (MUSAN). Частота дискретизации данных в рассматриваемой базе равна 16кГц по умолчанию.

3. Модернизировать загрузчик тренировочных данных под возможность случайного наложения (искажаем исходные звукозаписи) и не наложения (не искажаем исходные звукозаписи) одного из четырёх типов искажений (реверберация, музыкальный шум, фоновый шум голосов нескольких дикторов, неструктурированный шум), описанных внутри класса **AugmentWAV** следующего программного кода: **../common/DatasetLoader.py**.

4. Используя процедуру обучения из предыдущего пункта с идентичными настройками выполнить тренировку параметров блока генерации дикторских моделей на исходных данных при наличии их аугментирвоанных копий.

In [ ]:
# Загружаем MUSAN и базу импульсных откликов RIR
with AUGMENTATION_DATASET_LIST_PATH.open('r', encoding=TEXT_ENCODING) as dataset_list_file:
    augmentation_dataset_lines = dataset_list_file.readlines()

download_dataset(
    augmentation_dataset_lines,
    user=None,
    password=None,
    save_path=str(DATA_DIR),
)

In [ ]:
# Распаковываем MUSAN и необходимые части RIR
extract_dataset(save_path=str(DATA_DIR), fname=str(MUSAN_ARCHIVE_PATH))
part_extract(
    save_path=str(DATA_DIR),
    fname=str(RIR_ARCHIVE_PATH),
    target=list(RIR_EXTRACTION_TARGETS),
)

In [ ]:
# Разбиваем длинные записи MUSAN на короткие фрагменты
split_musan(save_path=str(DATA_DIR))

In [ ]:
# Создаём новую модель для обучения с аугментацией
speaker_model = ResNet(
    BasicBlock,
    layers=layers,
    activation=activation,
    num_filters=num_filters,
    nOut=embedding_size,
    encoder_type=encoder_type,
    n_mels=n_mels,
    log_input=log_input,
)
loss_function = AAMSoftmaxLoss(
    nOut=embedding_size,
    nClasses=number_of_train_speakers,
    margin=margin,
    scale=scale,
)
main_model = MainModel(speaker_model, loss_function).to(device)

In [ ]:
# Создаём загрузчик тренировочных данных с онлайн-аугментацией
train_dataset = train_dataset_loader(
    train_list=train_list,
    max_frames=max_frames_train,
    train_path=str(train_path),
    augment=True,
    musan_path=str(MUSAN_PATH),
    rir_path=str(RIR_PATH),
)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size_train,
    pin_memory=pin_memory,
    num_workers=num_workers_train,
    shuffle=shuffle,
)

validation_dataset = test_dataset_loader(
    test_list=validation_list,
    max_frames=max_frames_eval,
    test_path=str(train_path),
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size_eval,
    num_workers=num_workers_eval,
)

In [ ]:
# Повторно инициализируем оптимизатор и планировщик для новой модели
optimizer = SGDOptimizer(
    main_model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)
scheduler = OneCycleLRScheduler(
    optimizer,
    pct_start=ONE_CYCLE_START_FRACTION,
    cycle_momentum=False,
    max_lr=learning_rate,
    div_factor=ONE_CYCLE_DIV_FACTOR,
    final_div_factor=ONE_CYCLE_FINAL_DIV_FACTOR,
    total_steps=max_epoch * len(train_loader),
)

In [ ]:
start_epoch = 0
if CHECKPOINT_RESUME_ENABLED:
    checkpoint_path = (
        DEFAULT_AUGMENTED_CHECKPOINT_DIR
        / CHECKPOINT_FILENAME_TEMPLATE.format(epoch=CHECKPOINT_RESUME_EPOCH)
    )
    start_epoch = loadParameters(main_model, optimizer, scheduler, checkpoint_path) + 1

# Обучаем модель с онлайн-аугментацией
for epoch in range(start_epoch, max_epoch):
    train_loss, train_accuracy = train_network(
        train_loader,
        main_model,
        optimizer,
        scheduler,
        epoch + 1,
        verbose=True,
    )
    print(
        EPOCH_RESULT_MESSAGE.format(
            epoch=epoch + 1,
            loss=train_loss,
            accuracy=train_accuracy,
        )
    )
    if (epoch + 1) % validation_interval == 0:
        _, validation_accuracy = test_network(validation_loader, main_model)
        print(
            VALIDATION_RESULT_MESSAGE.format(
                epoch=epoch + 1,
                accuracy=validation_accuracy,
            )
        )
        saveParameters(
            main_model,
            optimizer,
            scheduler,
            epoch,
            DEFAULT_AUGMENTED_CHECKPOINT_DIR,
        )

**4. Тестирование блока построения дикторских моделей**

Из литературы известно, что применение алгоритмов машинного обучения на практике требует использования трёх наборов данных: *тренировочное множество* (используется для обучения параметров модели), *валидационное множество* (используется для настройки гиперпараметров), *тестовое множество* (используется для итогового тестирования).

В рамках настоящего пункта предлагается выполнить итоговое тестирование блоков генерации дикторских моделей, обученных без аугментации и с аугментацией тренировочных данных, и сравнить полученные результаты. При проведении процедуры тестирования рекомендуется выбрать различное количество фреймов для тестовых звукозаписей, чтобы грубо понять то, как длительность фонограммы влияет на качество распознавания диктора.

В качестве целевой метрики предлагается использовать *долю правильных ответов*, то есть количество верно классифицированных объектов по отношению к общему количеству объектов тестового множества. Как и при проведении процедуры обучения и валидации, рассматриваемая процедура тестирования предполагает решение задачи идентификации диктора на закрытом множестве.

In [ ]:
# Создаём загрузчик итоговой тестовой выборки
test_dataset = test_dataset_loader(
    test_list=test_list,
    max_frames=max_frames_eval,
    test_path=str(train_path),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size_eval,
    num_workers=num_workers_eval,
)

In [ ]:
# Загружаем и проверяем модель без аугментации
checkpoint_path = DEFAULT_CHECKPOINT_DIR / CHECKPOINT_FILENAME_TEMPLATE.format(
    epoch=MAX_EPOCHS - 1,
)
loadParameters(main_model, optimizer, scheduler, checkpoint_path)
_, test_accuracy = test_network(test_loader, main_model)
print(
    FULL_TEST_RESULT_MESSAGE.format(
        model_name=MODEL_WITHOUT_AUGMENTATION,
        accuracy=test_accuracy,
    )
)

In [ ]:
# Загружаем и проверяем модель с аугментацией
checkpoint_path = (
    DEFAULT_AUGMENTED_CHECKPOINT_DIR
    / CHECKPOINT_FILENAME_TEMPLATE.format(epoch=MAX_EPOCHS - 1)
)
loadParameters(main_model, optimizer, scheduler, checkpoint_path)
_, test_accuracy = test_network(test_loader, main_model)
print(
    FULL_TEST_RESULT_MESSAGE.format(
        model_name=MODEL_WITH_AUGMENTATION,
        accuracy=test_accuracy,
    )
)

**5. Контрольные вопросы**

1. Что такое верификация и идентицикация диктора?

2. Что такое распознавание диктора на закрытом и открытом множестве?

3. Что такое текстозависимое и текстонезависимое распознавание диктора?

4. Описать схему обучения блока генерации дикторских моделей на основе нейронных сетей.

5. Описать основные компоненты, из которых состоит нейросетевой блок генерации дикторских моделей (фреймовый уровень, слой статистического пулинга, сегментный уровень, выходной слой).

6. Как устроены нейросетевые архитектуры на основе ResNet-блоков?

7. Что такое полносвязная нейронная сеть прямого распространения?

8. Как устроена стоимостная функция для обучения нейросетевого блока генерации дикторских моделей?

9. Что такое аугментация данных?

10. Что такое дикторский эмбеддинг и на каком уровне блока построения дикторских моделей он генерируется?

**6. Список литературы**

1. Bai Z., Zhang X.-L., Chen J. Speaker recognition based on deep learning: an overview // 	arXiv:2012.00931 [eess.AS] ([ссылка](https://arxiv.org/pdf/2012.00931.pdf)).

2. Hansen J.H.L., Hasan T. Speaker recognition by machines and humans: a tutorial review // IEEE Signal Processing Magazine, 2015. V. 32. № 6. P. 74–99 ([ссылка](https://www.researchgate.net/publication/282940395_Speaker_Recognition_by_Machines_and_Humans_A_tutorial_review)).